<a href="https://colab.research.google.com/github/AmirMohammad73/semantic_folding/blob/main/Semantic_Space_Construction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import spacy
from collections import Counter

# Load the English language model from SpaCy
nlp = spacy.load("en_core_web_sm")

# Read the corpus.txt file and remove line numbers
with open("/content/drive/MyDrive/semantic_folding/Bopenbook.txt", "r", encoding="utf-8") as file:
    corpus_text = "\n".join(line.split(",", 1)[1].strip() for line in file.readlines())

# Process the corpus text using SpaCy
doc = nlp(corpus_text)

# Extract noun phrases and verb phrases from the processed text
phrases = []
for chunk in doc.noun_chunks:
    phrases.append(chunk.text)
for chunk in doc:
    if "VP" in chunk.dep_:
        phrases.append(chunk.text)

# Calculate the frequency of each phrase
phrase_counts = Counter(phrases)

# Define a function to filter phrases and remove stop words from single-word phrases
def filter_phrases(phrase_counts):
    filtered_phrases = {}
    for phrase, frequency in phrase_counts.items():
        words = phrase.split()
        if len (phrase) > 1 and len(words) == 1 and not nlp(words[0])[0].is_stop:
            filtered_phrases[phrase] = frequency
        elif len(words) > 1:
            filtered_phrases[phrase] = frequency
    return filtered_phrases

# Filter and sort phrases by frequency in descending order
filtered_phrases = filter_phrases(phrase_counts)
sorted_phrases = sorted(filtered_phrases.items(), key=lambda x: x[1], reverse=True)

# Write the filtered and sorted phrases and their frequencies to phrases.txt
with open("/content/drive/MyDrive/semantic_folding/phrases.txt", "w", encoding="utf-8") as output_file:
    for phrase, frequency in sorted_phrases:
        output_file.write(f"{phrase}: {frequency}\n")

In [3]:
import csv
from collections import defaultdict

# Read phrases from phrases.txt and store them in a list
with open("/content/drive/MyDrive/semantic_folding/phrases.txt", "r", encoding="utf-8") as phrases_file:
    phrases = [line.split(":")[0].strip() for line in phrases_file.readlines()]

# Read contexts from corpus.txt and store them in a list
with open("/content/drive/MyDrive/semantic_folding/Bopenbook.txt", "r", encoding="utf-8") as corpus_file:
    contexts = [line.split(",", 1)[0].strip() for line in corpus_file.readlines()]

# Initialize a term-context matrix as a defaultdict of defaultdicts
term_context_matrix = defaultdict(lambda: defaultdict(int))

# Read the processed corpus text
with open("/content/drive/MyDrive/semantic_folding/Bopenbook.txt", "r", encoding="utf-8") as corpus_file:
    corpus_lines = corpus_file.readlines()

# Process the corpus and populate the term-context matrix
for idx, line in enumerate(corpus_lines):
    context_id, context_text = line.split(",", 1)
    context_id = context_id.strip()
    context_text = context_text.strip()
    for phrase in phrases:
        term_context_matrix[context_id][phrase] = context_text.count(phrase)

# Write the term-context matrix to a CSV file
with open("/content/drive/MyDrive/semantic_folding/term_context_matrix.csv", "w", encoding="utf-8", newline="") as csv_file:
    csv_writer = csv.writer(csv_file)

    # Write the header row with phrase names
    csv_writer.writerow(["Context ID"] + phrases)

    # Write data rows with context IDs and phrase occurrences
    for context_id in contexts:
        row_data = [term_context_matrix[context_id][phrase] for phrase in phrases]
        csv_writer.writerow([context_id] + row_data)

print("Term-context matrix has been successfully created and saved to term_context_matrix.csv.")

Term-context matrix has been successfully created and saved to term_context_matrix.csv.


In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
import csv
import seaborn as sns
import numpy as np


NUM_DIMENSIONS = 10  # Number of dimensions (can be changed)
NUM_CONTEXT = 20

# Read the term-context matrix from the CSV file
term_context_matrix = {}
with open("/content/drive/MyDrive/semantic_folding/term_context_matrix.csv", "r", encoding="utf-8") as csv_file:
    reader = csv.reader(csv_file)
    header = next(reader)
    for row in reader:
        context_id = row[0]
        context_data = list(map(int, row[1:]))
        term_context_matrix[context_id] = context_data

# Create a graph for the contexts based on phrase overlap
G = nx.Graph()
for context_id, context_data in term_context_matrix.items():
    G.add_node(context_id)
    for neighbor_id, neighbor_data in term_context_matrix.items():
        if context_id != neighbor_id:
            weight = sum(c1 * c2 for c1, c2 in zip(context_data, neighbor_data))
            # weight = sum((1 if c1 > 0 else 0 ) * (1 if c2 > 0 else 0 ) for c1, c2 in zip(context_data, neighbor_data))
            weight_normalized = weight / 20
            if weight_normalized > 0.1:
                G.add_edge(context_id, neighbor_id, weight=weight)

# Use force-directed layout to position the nodes based on edge weights
pos = nx.spring_layout(G, seed=200, center=(0.0, 0.0), k=NUM_DIMENSIONS / 2, weight='weight')

# Calculate figsize based on the number of dimensions
figsize = (10, 6)

# Extract edge weights to set edge widths
edge_weights = [data['weight'] for _, _, data in G.edges(data=True)]

plt.figure(figsize=figsize)
nx.draw(G, pos, with_labels=True, node_color="skyblue", node_size=1500, font_size=10, font_weight="bold",
        width=edge_weights, edge_cmap=plt.cm.YlGnBu, edge_color=edge_weights, font_color="black", alpha=0.9)


# edge_cmap=plt.cm.PuRd
# edge_cmap=plt.cm.RdYlBu
# edge_cmap=plt.cm.magma

# Add edge labels (weights)
# edge_labels = {(edge[0], edge[1]): f'{edge[2]["weight"]:.2f}' for edge in G.edges(data=True)}
# nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_color='red')

plt.title("Semantic Matrix Mapping")
# Save the plot
plt.savefig(f"/content/drive/MyDrive/semantic_folding/images/graph.png")

plt.show()



context_context_matrix = np.zeros((NUM_CONTEXT, NUM_CONTEXT), dtype=int)
for context_id, context_data in term_context_matrix.items():
    for neighbor_id, neighbor_data in term_context_matrix.items():
        if context_id != neighbor_id:
            weight = sum(c1 * c2 for c1, c2 in zip(context_data, neighbor_data))
            weight_normalized = weight / 20
            if weight_normalized > 0.1:
                context_context_matrix[int(context_id) - 1, int(neighbor_id) - 1] = int(weight)

# Adjusting the matrix for display
adjusted_matrix = np.flipud(context_context_matrix)

# Create the heatmap
plt.figure(figsize=(10, 8))  # You can adjust the figure size as per your preference
sns.heatmap(adjusted_matrix, annot=True, fmt="d", cmap="YlGnBu",
            xticklabels=list(range(1, NUM_CONTEXT + 1)), yticklabels=list(range(NUM_CONTEXT, 0, -1)))

# Adding title and labels if needed
plt.title('Context-Context Matrix Heatmap')
plt.xlabel('Context ID')
plt.ylabel('Neighbor ID')

# # Save the plot
plt.savefig(f"/content/drive/MyDrive/semantic_folding/images/context_context_heatmap.png")
# Show the plot
plt.show()

# Generate a dictionary to store context IDs and their corresponding cell coordinates
context_coordinates = {}
for context_id, position in pos.items():
    row, col = int(position[1] * NUM_DIMENSIONS / 2), int(position[0] * NUM_DIMENSIONS / 2)
    context_coordinates[context_id] = f"{row},{col}"

# Write context IDs and their cell coordinates to a CSV file
with open("/content/drive/MyDrive/semantic_folding/context_coordinates.csv", "w", encoding="utf-8", newline="") as csv_file:
    csv_writer = csv.writer(csv_file)
    csv_writer.writerow(["Context ID", "Cell Coordinates"])
    for context_id, coordinates in context_coordinates.items():
        csv_writer.writerow([context_id, coordinates])

print("Context coordinates have been successfully saved to context_coordinates.csv.")

# Extract row and column coordinates from context_coordinates
rows = []
cols = []
for coordinates in context_coordinates.values():
    row, col = map(int, coordinates.split(','))
    rows.append(row)
    cols.append(col)

# Create a NumPy array for the matrix
matrix_data = np.zeros((NUM_DIMENSIONS, NUM_DIMENSIONS), dtype=int)
# Adjusting the matrix for display
adjusted_matrix_data = np.flipud(matrix_data)


for row, col in zip(rows, cols):
    matrix_data[row, col] += 1
plt.figure(figsize=(8, 6))
# Create a heatmap using seaborn
sns.heatmap(adjusted_matrix_data, annot=True, fmt="d", cmap="YlGnBu", cbar=True,
            xticklabels=list(range(1, NUM_DIMENSIONS + 1)), yticklabels=list(range(NUM_DIMENSIONS, 0, -1)))
# plt.scatter(cols, rows, marker='s', color='red', label='Contexts')  # Add red squares for context positions
plt.title("Semantic Matrix Mapping - Heatmap")
plt.xlabel("Column")
plt.ylabel("Row")
plt.legend()
plt.savefig(f"/content/drive/MyDrive/semantic_folding/images/semantic_space_heatmap.png")
plt.show()


import plotly.graph_objs as go

context_text_mapping = {}

with open("/content/drive/MyDrive/semantic_folding/Bopenbook.txt", "r", encoding="utf-8") as file:
    for line in file:
        # Assuming each line is in the format: "line_number,context_text"
        line_number, context_text = line.strip().split(',', 1)
        context_text_mapping[line_number] = line # context_text

# Create an empty array for the line numbers
line_numbers = [["" for _ in range(NUM_DIMENSIONS)] for _ in range(NUM_DIMENSIONS)]
# Create an empty array for the heatmap
context_texts = [["" for _ in range(NUM_DIMENSIONS)] for _ in range(NUM_DIMENSIONS)]

# Populate the array with line numbers and context texts
for context_id, coordinates in context_coordinates.items():
    row, col = map(int, coordinates.split(','))
    current_lines = line_numbers[row][col]
    new_line = context_id  # Using context_id as the line number

    # Concatenate if there's already a line number in this cell
    if current_lines:
        line_numbers[row][col] = current_lines + ", " + new_line
    else:
        line_numbers[row][col] = new_line

    # Assign context text to the corresponding cell
    current_text = context_texts[row][col]
    new_text = context_text_mapping.get(context_id, "")

    # Concatenate if there's already text in this cell
    if current_text:
        context_texts[row][col] = current_text + " || " + new_text
    else:
        context_texts[row][col] = new_text

# Adjust the matrices for display
adjusted_line_numbers = np.flipud(line_numbers)

adjusted_context_texts = np.flipud(context_texts)

# Create a figure for line numbers using Seaborn
plt.figure(figsize=(8, 6))
sns.heatmap(np.zeros_like(adjusted_matrix_data), annot=adjusted_line_numbers, fmt="", cmap="YlGnBu", cbar=False,
            xticklabels=list(range(1, NUM_DIMENSIONS + 1)), yticklabels=list(range(1, NUM_DIMENSIONS + 1)))
plt.title("Semantic Matrix Mapping - Line Numbers")
plt.xlabel("Column")
plt.ylabel("Row")
plt.savefig(f"/content/drive/MyDrive/semantic_folding/images/semantic_space_heatmap_context_id.png")

plt.show()



trace = go.Heatmap(
    z=adjusted_matrix_data,
    x=list(range(1, NUM_DIMENSIONS + 1)),
    y=list(range(NUM_DIMENSIONS, 0, -1)),
    text=adjusted_line_numbers,  # Display line numbers in each cell
    hovertext=adjusted_context_texts,  # Show context information on hover
    hoverinfo="text",
    colorscale="YlGnBu"
)

# Layout configuration
layout = go.Layout(
    title="Semantic Matrix Mapping - Heatmap",
    xaxis=dict(title="Column"),
    yaxis=dict(title="Row"),
    hovermode='closest'
)

print(adjusted_line_numbers)
# Create figure
fig = go.Figure(data=[trace], layout=layout)

# Show the figure
fig.show()

In [ ]:
import csv
import os

NUM_DIMENSIONS = 8  # Number of dimensions (can be changed)

# Load context coordinates from context_coordinates.csv
context_coordinates = {}
with open("/content/drive/MyDrive/semantic_folding/context_coordinates.csv", "r", encoding="utf-8") as csv_file:
    reader = csv.reader(csv_file)
    header = next(reader)
    for row in reader:
        context_id, coordinates = row
        context_coordinates[context_id] = tuple(map(int, coordinates.split(',')))

# Load phrases from phrases.txt and map them to their indices in term_context_matrix header
phrase_indices = {}
phrases = []
with open("/content/drive/MyDrive/semantic_folding/phrases.txt", "r", encoding="utf-8") as file:
    for idx, line in enumerate(file):
        phrase = line.strip()
        phrases.append(phrase)
        phrase_indices[phrase] = idx + 1  # Adding 1 because the first column in term_context_matrix is context_id

# Initialize fingerprints directory
if not os.path.exists("fingerprints"):
    os.makedirs("fingerprints")

# Load term-context matrix from term_context_matrix.csv
term_context_matrix = {}
with open("/content/drive/MyDrive/semantic_folding/term_context_matrix.csv", "r", encoding="utf-8") as csv_file:
    reader = csv.reader(csv_file)
    header = next(reader)
    for row in reader:
        context_id = row[0]
        context_data = list(map(int, row[1:]))
        term_context_matrix[context_id] = context_data

# Generate and save fingerprints for each phrase
for phrase in phrases:
    print("*-"*30)
    print(phrase)
    fingerprint_matrix = [[0 for _ in range(NUM_DIMENSIONS)] for _ in range(NUM_DIMENSIONS)]
    phrase_index = phrase_indices[phrase]
    print(f"Phrase Index:{phrase_index}")
    for context_id, context_data in term_context_matrix.items():
        if phrase_index < len(phrases):
            for i, item in enumerate(context_data[:]):
                if item> 0 :
                    print(phrases[i])
            if context_data[phrase_index] == 1:
                print(f"===========  Phrase Matched, Context ID : {context_id} ==========")
                row, col = context_coordinates[context_id]
                print(f"||>>>> Semantic Coordinates: {row}, {col}")
                # fingerprint_matrix[(row+NUM_DIMENSIONS)//2][(col+NUM_DIMENSIONS)//2] += 1
                fingerprint_matrix[row][col] += 1



    print(fingerprint_matrix)
    # Save fingerprint matrix to a file
    fingerprint_filename = os.path.join("fingerprints", f"{phrase.split(':')[0].replace(' ', '_')}_fingerprint.txt")
    print(f"File Name : {fingerprint_filename}")
    # input()
    with open(fingerprint_filename, "w", encoding="utf-8") as fingerprint_file:
        for row in fingerprint_matrix:
            fingerprint_file.write("\t".join(map(str, row)) + "\n")

print("Fingerprints have been successfully generated and saved in the 'fingerprints' folder.")

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Function to load fingerprint matrix from file
def load_fingerprint_matrix(phrase):
    fingerprint_filename = os.path.join("fingerprints", f"{phrase.replace(' ', '_')}_fingerprint.txt")
    if not os.path.exists(fingerprint_filename):
        print(f"Fingerprint file for '{phrase}' not found.")
        return None
    with open(fingerprint_filename, "r", encoding="utf-8") as file:
        fingerprint_matrix = [[int(value) for value in line.strip().split("\t")] for line in file]
    return fingerprint_matrix

# Function to plot fingerprint matrix as heatmap using Seaborn for a single phrase
def plot_single_phrase_heatmap(phrase, fingerprint_matrix):
    plt.figure(figsize=(8, 8))
    sns.heatmap(fingerprint_matrix, annot=True, fmt="d", cmap="Blues", cbar=False, xticklabels=True, yticklabels=True)
    plt.title(f"{phrase} Fingerprint Heatmap")
    plt.xlabel("Columns")
    plt.ylabel("Rows")

    # Save the plot
    plt.savefig(f"./images/phrase_fingerprint_heatmap_{phrase.replace(' ', '_')}.png")

    plt.show()

# Function to plot fingerprint matrices as side-by-side heatmaps using Seaborn for two phrases
def plot_two_phrases_heatmaps(phrase1, fingerprint_matrix1, phrase2, fingerprint_matrix2):
    plt.figure(figsize=(16, 8))

    plt.subplot(1, 2, 1)
    sns.heatmap(fingerprint_matrix1, annot=True, fmt="d", cmap="Blues", cbar=False, xticklabels=True, yticklabels=True)
    plt.title(f"{phrase1} Fingerprint Heatmap")
    plt.xlabel("Columns")
    plt.ylabel("Rows")

    plt.subplot(1, 2, 2)
    sns.heatmap(fingerprint_matrix2, annot=True, fmt="d", cmap="Blues", cbar=False, xticklabels=True, yticklabels=True)
    plt.title(f"{phrase2} Fingerprint Heatmap")
    plt.xlabel("Columns")
    plt.ylabel("Rows")

    # Save the plot
    plt.savefig(f"./images/phrase_fingerprint_heatmap_{phrase1.replace(' ', '_')}_{phrase2.replace(' ', '_')}.png")

    plt.show()

# Main function
def main():
    phrases = input("Enter one or two phrases (comma separated): ").split(",")
    phrases = [phrase.strip() for phrase in phrases]

    if len(phrases) == 1:
        phrase1 = phrases[0]
        fingerprint_matrix1 = load_fingerprint_matrix(phrase1)
        if fingerprint_matrix1:
            plot_single_phrase_heatmap(phrase1, fingerprint_matrix1)
        else:
            print(f"Fingerprint matrix not found for '{phrase1}'.")
    elif len(phrases) == 2:
        phrase1, phrase2 = phrases
        fingerprint_matrix1 = load_fingerprint_matrix(phrase1)
        fingerprint_matrix2 = load_fingerprint_matrix(phrase2)
        if fingerprint_matrix1 and fingerprint_matrix2:
            plot_two_phrases_heatmaps(phrase1, fingerprint_matrix1, phrase2, fingerprint_matrix2)
        else:
            print(f"Fingerprint matrix not found for one or both phrases.")
    else:
        print("Invalid number of phrases. Please enter one or two phrases.")

if __name__ == "__main__":
    main()

In [ ]:
import os
import spacy
import numpy as np
from collections import Counter

# Load the English language model from SpaCy
nlp = spacy.load("en_core_web_sm")

# Function to extract phrases from a sentence using SpaCy
def extract_phrases(sentence):
    doc = nlp(sentence)
    phrases = [chunk.text for chunk in doc.noun_chunks]
    print("Processing Sentence : ")
    print(f"\t{phrases}")
    print(f"Phrases : ")
    print(",".join(phrases))
    return phrases

# Function to load fingerprint matrix from file
def load_fingerprint_matrix(phrase):
    fingerprint_filename = os.path.join("fingerprints", f"{phrase.replace(' ', '_')}_fingerprint.txt")
    if not os.path.exists(fingerprint_filename):
        print(f"Fingerprint file for '{phrase}' not found.")
        return None
    with open(fingerprint_filename, "r", encoding="utf-8") as file:
        fingerprint_matrix = np.array([[int(value) for value in line.strip().split("\t")] for line in file])
    return fingerprint_matrix

# Function to generate document fingerprint by summing phrase fingerprints
def generate_doc_fingerprint(sentence, phrase_fingerprints):
    # doc_fingerprint = np.zeros_like(phrase_fingerprints[0])
    doc_fingerprint = np.zeros_like(next(iter(phrase_fingerprints.values())))

    phrases = extract_phrases(sentence)
    print("|||>>>>>>> Doc Fingerprints ")
    for phrase in phrases:
        if phrase in phrase_fingerprints:
            doc_fingerprint += phrase_fingerprints[phrase]
            print(doc_fingerprint)

    return doc_fingerprint


# Main function
def main():
    # Create a new folder for document fingerprints if it doesn't exist
    doc_fingerprints_folder = "doc_fingerprints"
    os.makedirs(doc_fingerprints_folder, exist_ok=True)

    # Load phrase fingerprints into a dictionary
    phrase_fingerprints = {}
    for phrase_filename in os.listdir("fingerprints"):
        phrase = phrase_filename.split("_fingerprint")[0].replace('_', ' ')
        fingerprint_matrix = load_fingerprint_matrix(phrase)
        if fingerprint_matrix is not None:
            phrase_fingerprints[phrase] = fingerprint_matrix

    # Read the corpus.txt file and generate document fingerprints
    with open("corpus.txt", "r", encoding="utf-8") as corpus_file:
        for line_number, sentence in enumerate(corpus_file, start=1):
            sentence = sentence.strip()
            doc_fingerprint = generate_doc_fingerprint(sentence, phrase_fingerprints)

             # Save document fingerprint to file
            phrases_in_sentence = '__'.join(extract_phrases(sentence))
            phrases_in_sentence = phrases_in_sentence.replace(',', '').replace('.', '')
            phrase_name = f"doc_{line_number}_{phrases_in_sentence.replace(' ', '_')}"
            doc_filename = os.path.join(doc_fingerprints_folder, f"{phrase_name}_fingerprint.txt")
            np.savetxt(doc_filename, doc_fingerprint, delimiter='\t', fmt='%d')
            # if line_number == 2 :
            #     break
    print("Document fingerprints have been generated and saved in the 'doc_fingerprints' folder.")

if __name__ == "__main__":
    main()